# 00b — Verificación del Manifest (≥50 GB + hashes MD5)

Este notebook carga la lista de archivos desde el bucket Azure ADLS Gen2,
calcula o verifica los hashes MD5, suma el tamaño total y genera el
`manifest.json` versionable.

**Requisito:** credenciales de Azure en variables de entorno o archivo `.env`
```
AZURE_STORAGE_ACCOUNT_NAME=stanaliticafinal
AZURE_STORAGE_ACCOUNT_KEY=<tu_key>
AZURE_CONTAINER=geovision
```


In [1]:
import subprocess, sys
for pkg in ['azure-storage-blob', 'azure-identity', 'python-dotenv']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)
print('Dependencias listas.')

Dependencias listas.


In [2]:
import os, json, hashlib
from pathlib import Path
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.blob import BlobServiceClient
from tqdm.auto import tqdm

# Cargar .env
load_dotenv()

TENANT_ID     = os.getenv('AZURE_TENANT_ID', '')
CLIENT_ID     = os.getenv('AZURE_CLIENT_ID', '')
CLIENT_SECRET = os.getenv('AZURE_CLIENT_SECRET', '')
ACCOUNT_NAME  = os.getenv('AZURE_STORAGE_ACCOUNT', 'stanaliticafinal')
CONTAINER     = os.getenv('AZURE_CONTAINER', 'geovision')

OUTPUT_DIR   = Path(r'D:\analitica')
MANIFEST_OUT = OUTPUT_DIR / 'manifest.json'

# ── Autenticación con Service Principal (Client ID + Secret) ─────────────
# Este es el método correcto para tu .env con AZURE_CLIENT_ID y AZURE_CLIENT_SECRET
missing = [k for k, v in [('AZURE_TENANT_ID', TENANT_ID),
                            ('AZURE_CLIENT_ID', CLIENT_ID),
                            ('AZURE_CLIENT_SECRET', CLIENT_SECRET)] if not v]
if missing:
    print(f'[ERROR] Faltan variables en .env: {missing}')
    print('Completa el .env con los valores del Azure Portal:')
    print('  AZURE_TENANT_ID     → Azure Portal → Entra ID → Overview → Tenant ID')
    print('  AZURE_CLIENT_ID     → Entra ID → App Registrations → tu app → Application (client) ID')
    print('  AZURE_CLIENT_SECRET → Entra ID → App Registrations → tu app → Certificates & secrets')
else:
    credential = ClientSecretCredential(
        tenant_id=TENANT_ID,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    )
    account_url   = f'https://{ACCOUNT_NAME}.blob.core.windows.net'
    blob_service  = BlobServiceClient(account_url=account_url, credential=credential)
    container_client = blob_service.get_container_client(CONTAINER)

    # Verificar conexión
    try:
        props = container_client.get_container_properties()
        print(f'✓ Conectado: {ACCOUNT_NAME}/{CONTAINER}')
        print(f'  Last modified: {props.last_modified}')
    except Exception as e:
        print(f'[ERROR] No se pudo conectar: {e}')
        print('Verifica que el Service Principal tiene rol "Storage Blob Data Reader" en el container')


✓ Conectado: stanaliticafinal/geovision
  Last modified: 2026-05-04 01:17:53+00:00


In [3]:
# ── Listar todos los blobs del container ─────────────────────────────────
print('Listando archivos en el bucket...')

blobs_info = []
total_bytes = 0

for blob in tqdm(container_client.list_blobs(), desc='Listando blobs'):
    blobs_info.append({
        'nombre'         : blob.name,
        'tamaño_bytes'   : blob.size,
        'tamaño_GB'      : round(blob.size / 1e9, 4),
        'ultima_modificacion': str(blob.last_modified),
        'content_type'   : blob.content_settings.content_type if blob.content_settings else '',
    })
    total_bytes += blob.size

total_GB = total_bytes / 1e9
print(f'\nTotal blobs: {len(blobs_info)}')
print(f'Tamaño total: {total_GB:.2f} GB')
print(f'Gate ≥50 GB: {"[OK] ✓" if total_GB >= 50 else "[FAIL] ✗"}')

df_blobs = pd.DataFrame(blobs_info)
print(f'\nDesglose por extensión:')
df_blobs['ext'] = df_blobs['nombre'].apply(lambda x: Path(x).suffix.lower())
print(df_blobs.groupby('ext')['tamaño_GB'].sum().sort_values(ascending=False).head(10))

Listando archivos en el bucket...


Listando blobs: 0it [00:00, ?it/s]


Total blobs: 21311
Tamaño total: 306.54 GB
Gate ≥50 GB: [OK] ✓

Desglose por extensión:
ext
.tif     292.9435
.hdf      12.6831
.csv       0.8081
.nc        0.0648
.yaml      0.0010
.log       0.0003
.json      0.0002
           0.0000
Name: tamaño_GB, dtype: float64


In [4]:
# ── Calcular hashes MD5 (streaming, sin descargar a disco) ───────────────
# Para datasets grandes (>1 GB) el hash se calcula por streaming.
# Para reducir tiempo, solo calcular MD5 de archivos clave (zarr, parquet, json).

KEY_EXTENSIONS = {'.zarr', '.parquet', '.json', '.jsonl', '.pt', '.csv'}
KEY_BLOBS = [b for b in blobs_info
             if Path(b['nombre']).suffix.lower() in KEY_EXTENSIONS
             or b['tamaño_bytes'] > 100_000_000]   # o archivos >100 MB

print(f'Calculando MD5 para {len(KEY_BLOBS)} archivos clave...')

CHUNK = 4 * 1024 * 1024   # 4 MB por chunk

for b in tqdm(KEY_BLOBS, desc='MD5 hashing'):
    try:
        blob_client = container_client.get_blob_client(b['nombre'])
        # Descargar en chunks para no cargar todo en memoria
        stream = blob_client.download_blob()
        h = hashlib.md5()
        for chunk in stream.chunks():
            h.update(chunk)
        b['md5'] = h.hexdigest()
    except Exception as e:
        b['md5'] = f'ERROR: {e}'

print('MD5 calculados.')

Calculando MD5 para 1507 archivos clave...


MD5 hashing:   0%|          | 0/1507 [00:00<?, ?it/s]

MD5 calculados.


In [5]:
# ── Generar manifest.json ─────────────────────────────────────────────────
manifest = {
    'proyecto'      : 'GeoVision-CLIP Cali',
    'fecha_generacion': datetime.now().isoformat(),
    'bucket'        : f'{ACCOUNT_NAME}/{CONTAINER}',
    'total_blobs'   : len(blobs_info),
    'total_GB'      : round(total_GB, 4),
    'gate_50GB'     : bool(total_GB >= 50),
    'archivos'      : blobs_info,
}

with open(MANIFEST_OUT, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=str)

print(f'Manifest guardado: {MANIFEST_OUT}')
print(f'Tamaño manifest: {MANIFEST_OUT.stat().st_size:,} B')

# MD5 del propio manifest
manifest_md5 = hashlib.md5(MANIFEST_OUT.read_bytes()).hexdigest()
print(f'MD5 manifest: {manifest_md5}')

# Resumen para el informe
print('\n=== Resumen para el informe ===')
print(f'Total datos en bucket: {total_GB:.2f} GB')
print(f'Gate ≥50 GB: {"[OK] ✓" if total_GB >= 50 else "[FAIL] ✗"}')
print(f'Total archivos versionados: {len(blobs_info)}')
print(f'Archivos con MD5 verificado: {sum(1 for b in blobs_info if b.get("md5") and not str(b.get("md5","")).startswith("ERROR"))}')

# Tabla de los archivos más grandes
df_top = df_blobs.nlargest(10, 'tamaño_GB')
print('\nTop 10 archivos por tamaño:')
print(df_top[['nombre','tamaño_GB']].to_string(index=False))

Manifest guardado: D:\analitica\manifest.json
Tamaño manifest: 5,020,841 B
MD5 manifest: 9a4fe1c3499042548d2082bf1f942bf1

=== Resumen para el informe ===
Total datos en bucket: 306.54 GB
Gate ≥50 GB: [OK] ✓
Total archivos versionados: 21311
Archivos con MD5 verificado: 1507

Top 10 archivos por tamaño:
                                    nombre  tamaño_GB
sentinel2/S2B_18NUK_20260214_0_L2A/B08.tif     0.1922
sentinel2/S2B_18NUK_20260316_0_L2A/B08.tif     0.1893
sentinel2/S2A_18NUK_20260313_1_L2A/B08.tif     0.1880
sentinel2/S2C_18NUK_20260420_0_L2A/B08.tif     0.1851
sentinel2/S2C_18NUK_20260321_0_L2A/B08.tif     0.1840
sentinel2/S2C_18NUK_20260331_0_L2A/B08.tif     0.1835
sentinel2/S2A_18NUK_20200112_0_L2A/B08.tif     0.1827
sentinel2/S2B_18NUK_20200206_0_L2A/B08.tif     0.1823
sentinel2/S2A_18NUK_20200322_0_L2A/B08.tif     0.1822
sentinel2/S2A_18NUK_20200322_1_L2A/B08.tif     0.1814


In [6]:
# ── Alternativa local: verificar archivos descargados en disco ────────────
# Si no tienes acceso al bucket en este momento, puedes verificar
# los archivos locales con este bloque.

LOCAL_DIRS = [
    Path(r'D:\analitica\procesado_zarr'),
    Path(r'D:\analitica\outputs_geovision'),
    Path(r'D:\analitica'),
]

local_files = []
for d in LOCAL_DIRS:
    if d.exists():
        for p in d.rglob('*'):
            if p.is_file() and p.suffix.lower() in {'.zarr','.parquet','.json',
                                                     '.jsonl','.pt','.csv','.nc'}:
                local_files.append({
                    'ruta': str(p),
                    'tamaño_GB': round(p.stat().st_size / 1e9, 4),
                    'nombre': p.name,
                })

total_local_GB = sum(f['tamaño_GB'] for f in local_files)
print(f'Archivos locales encontrados: {len(local_files)}')
print(f'Tamaño total local: {total_local_GB:.2f} GB')
print(f'Gate ≥50 GB (local): {"[OK] ✓" if total_local_GB >= 50 else "[REVISAR] — verificar bucket"}')

df_local = pd.DataFrame(local_files)
if not df_local.empty:
    print('\nTop archivos locales:')
    print(df_local.nlargest(10,'tamaño_GB')[['nombre','tamaño_GB']].to_string(index=False))

Archivos locales encontrados: 62
Tamaño total local: 8.30 GB
Gate ≥50 GB (local): [REVISAR] — verificar bucket

Top archivos locales:
                 nombre  tamaño_GB
       RS5M_ViT-H-14.pt     3.9447
   RS5M_ViT-L-14-336.pt     1.7119
       RS5M_ViT-B-32.pt     0.6052
 RS5M_ViT-B-32_RET-2.pt     0.6052
RS5M_ViT-B-32_RSITMD.pt     0.6052
    late_fusion_best.pt     0.0727
    late_fusion_best.pt     0.0727
    late_fusion_best.pt     0.0727
    late_fusion_best.pt     0.0727
    late_fusion_best.pt     0.0727
